# Data Wrangling: Normalizing Compensation Data

Part of my IBM Data Analyst Capstone project. In this notebook I remove duplicates, fill missing values in `CodingActivities`, and normalise the `ConvertedCompYearly` compensation column using **Min-Max scaling** and **Z-score standardisation**, then compare the distributions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load the data

In [1]:
file_path ="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UDKAZw-kz18Yj8P6icf_qw/survey-data-duplicates.csv"

df = pd.read_csv(file_path)

# Display the first few rows
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

## 2. Remove duplicate rows

In [2]:
# Task 1: Identify and remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Duplicate rows: 10Shape after removing duplicates: (65437, 114)

## 3. Handle missing values in CodingActivities

In [3]:
# Task 2: Identify missing values in CodingActivities
print("Missing in CodingActivities:", df['CodingActivities'].isnull().sum())

Missing in CodingActivities: 10971

I filled the gaps with **forward-fill**, which copies the previous row's value into each gap. Forward-fill suits ordered data such as time series. In survey data each row is a different person, so it copies one respondent's answer onto another. It is shown here as a technique; for analysis I would prefer the most frequent value or leave the gaps missing.

In [4]:
# Task 3: Impute missing values with forward-fill
df['CodingActivities'] = df['CodingActivities'].ffill()

# Verify
print("Missing after ffill:", df['CodingActivities'].isnull().sum())

Missing after ffill: 0

## 4. Identify compensation columns
The survey has two compensation columns: `CompTotal` (as reported, in local currencies) and `ConvertedCompYearly` (converted to annual USD). I use `ConvertedCompYearly`.

About 64% of respondents did not report compensation. As in the wrangling notebook, I did not fill those gaps with the median, because that would give most rows an invented salary. I normalise only respondents who reported pay.

In [ ]:
comp_cols = [c for c in df.columns if c.startswith(('Comp', 'ConvertedComp'))]
print(comp_cols)

missing = df['ConvertedCompYearly'].isnull().sum()
print("Missing in ConvertedCompYearly:", missing, f"({missing / len(df):.1%})")

# Keep only respondents who reported compensation
comp_df = df.dropna(subset=['ConvertedCompYearly']).copy()
print("Rows with reported compensation:", len(comp_df))

## 5. Min-Max scaling
Rescales values to the range 0 to 1 using (x - min) / (max - min).

In [ ]:
col = comp_df['ConvertedCompYearly']
comp_df['ConvertedCompYearly_MinMax'] = (col - col.min()) / (col.max() - col.min())

print(comp_df['ConvertedCompYearly_MinMax'].describe())

## 6. Z-score standardisation
Rescales values to a mean of 0 and a standard deviation of 1 using (x - mean) / std.

In [ ]:
comp_df['ConvertedCompYearly_Zscore'] = (col - col.mean()) / col.std()

print(comp_df['ConvertedCompYearly_Zscore'].describe())

## 7. Compare the distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(comp_df['ConvertedCompYearly'], bins=50, color='steelblue')
axes[0].set_title('Original ConvertedCompYearly')
axes[0].set_xlabel('Yearly compensation (USD)')

axes[1].hist(comp_df['ConvertedCompYearly_MinMax'], bins=50, color='seagreen')
axes[1].set_title('Min-Max Normalized')
axes[1].set_xlabel('Scaled value (0 to 1)')

axes[2].hist(comp_df['ConvertedCompYearly_Zscore'], bins=50, color='darkorange')
axes[2].set_title('Z-score Normalized')
axes[2].set_xlabel('Z-score')

for ax in axes:
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

The shape of the data does not change, only the scale. Compensation has extreme outliers (the highest value is about 16 million, against a median near 65,000), so Min-Max scaling squeezes most values close to 0 and Z-scores show a few very large values. Outliers would need handling before using normalised compensation for modelling.